# Project 6: Change control

## 1. What can happen?

A prompt can change after the application was evaluated and approved.

**Example** Someone adds a sentence to the prompt but tries to reuse the old approval.

## 2. Our idea

We make a release manifest: a record of code, model configuration, prompt, dataset, and policy identities. SHA-256 gives each file a content fingerprint. Fixed fictional evaluation and approval records refer to the bundled baseline manifest. The notebook never regenerates those records.

A separate change note records why the prompt was modified, the expected
behavior, and planned retests. Its AI-authored example intent awaits human
review. Actual retest results start empty and pending.

### How we do it

1. Recalculate the baseline manifest from the actual files.
2. Check its binding to the saved fictional records.
3. Select the candidate prompt file, recalculate, and show why the old records no longer match.
4. Read the linked change note and distinguish planned retests from recorded results.

## 3. How it helps

The prompt is the only changed artifact. Both previous records stop matching. Hashes detect content changes; they do not prove report quality or authenticate a reviewer. This lesson checks version links and change records, not model quality or full release eligibility.

## 4. Input and setup

Run uv sync --locked at the outer root. Open this numbered folder in VS Code,
select the root .venv/Scripts/python.exe kernel, and run one cell at a time.
All source examples are fictional and make no model calls.


### Step 1: Check the lesson folder

This cell only checks the working directory and imports standard-library tools.

In [ ]:
import json
from copy import deepcopy
from datetime import date
from pathlib import Path

project_dir = Path.cwd()
assert project_dir.name == "6_change_control", "Open this numbered folder in VS Code."
as_of = date(2026, 11, 15)
print("Lesson:", project_dir.name, "| fixed sample date:", as_of)

### Step 2: Inspect the fixed records

These saved records were authored for the bundled baseline once. They are not created from whatever prompt happens to be present when the notebook runs.

In [ ]:
release = json.loads((project_dir / "docs/dataset/release.json").read_text())
records = json.loads((project_dir / "docs/dataset/release_records.json").read_text())
saved_manifest = json.loads((project_dir / "docs/dataset/baseline_manifest.json").read_text())
print(json.dumps(records, indent=2))

### Step 3: Recalculate the actual baseline

Read tools/manifest.py. build_manifest reads each named artifact and hashes a stable manifest. check_bindings compares the fixed evaluation and approval identities with that computed manifest.

In [ ]:
from tools.manifest import build_manifest, check_bindings

baseline_manifest = build_manifest(project_dir, release)
baseline = check_bindings(baseline_manifest, records)
print("Bundled baseline unchanged:", baseline_manifest == saved_manifest)
print(baseline)

### Step 4: Inspect the one prompt change

Both files are already supplied. Selecting the candidate never overwrites the baseline prompt.

In [ ]:
print("Baseline:", (project_dir / "prompts/answer.txt").read_text())
print("Candidate:", (project_dir / "prompts/candidate_answer.txt").read_text())

### Step 5: Build the candidate manifest

Change only the prompt selection. Use exactly the same saved evidence and fictional approval.

In [ ]:
candidate_release = deepcopy(release)
candidate_release["artifacts"]["prompt"] = "prompts/candidate_answer.txt"
candidate_manifest = build_manifest(project_dir, candidate_release)
candidate = check_bindings(candidate_manifest, records)
print(candidate)

### Step 6: Compare artifact identities

Only the prompt should differ. Restoring matching bytes can restore binding, but it does not supply a real approval or validate the evidence.

In [ ]:
for kind in baseline_manifest["artifacts"]:
    changed = baseline_manifest["artifacts"][kind] != candidate_manifest["artifacts"][kind]
    print(kind, "changed:", changed)
print(baseline["decision"], "->", candidate["decision"])

### Step 7: Read the intent behind the change

Read [the change note](docs/dataset/change_note.json) and
[tools/change_notes.py](tools/change_notes.py) before importing the helper.
The note links to these exact before/after manifest IDs. Its reason and expected
behavior are example draft text; a real change needs the actual author's account.

check_change_note checks version links, narrative fields, unique planned case
IDs, and consistency between pending/completed status and recorded results.
It does not authenticate an author, verify evidence contents, or approve a release.

The plan keeps Q-1 and proposes GAP-1, a parental-leave question that the policy
does not answer. GAP-1 is a proposed retest, not part of the baseline dataset.
Expect matching links, no note issues, and pending retests with no results.
The candidate should still need evaluation and human review.


In [ ]:
from tools.change_notes import check_change_note

change_note = json.loads((project_dir / "docs/dataset/change_note.json").read_text())
note_check = check_change_note(change_note, baseline_manifest, candidate_manifest)
print("Why:", change_note["reason"])
print("Expected behavior:", change_note["expected_behavior"])
print("Planned cases:", [case["case_id"] for case in change_note["planned_retests"]])
print("Recorded results:", change_note["retest_results"])
print("Human review:", change_note["human_review"])
print(note_check)

### Explain the result

The baseline gives bindings_match. The candidate gives needs_evaluation_and_review. Only the prompt artifact differs, and the stored baseline manifest and fictional records remain unchanged.

The prompt is the only changed artifact. Both previous records stop matching. Hashes detect content changes; they do not prove report quality or authenticate a reviewer. This lesson checks version links and change records, not model quality or full release eligibility.

The linked change note adds the reason, intended behavior, and retest plan.
It does not change the candidate's need for evaluation and review. No model
retests have run: results remain empty and human review is pending.

After actual retesting, record case_id, outcome (pass/fail), observation, and
evidence_ref for each planned case. Mark retests completed only when every
planned result is recorded. A completed retest may fail; it is not approval.
A reviewer still needs to inspect the referenced evidence.

Changes are in memory. This notebook writes no files.

Pause here. Your run and any human decisions remain your learning steps.